# Portfolio Risk

A portfolio of a pay-fixed swap, fixed-rate bond and caplet. Instruments hold contractual amounts and terms; explicit trades hold execution dates, direction, position-size overrides and bond pricing anchors.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import quantsupport as qs
from market_data import load_market

market = load_market()
ref = market.quotes.reference_date
swap = qs.Swap(
    identifier="BOOK-SWAP",
    start_date=ref,
    maturity_date=ref + "5Y",
    notional=1_000_000,
    fixed_rate=0.04,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
)
bond = qs.FixedRateBond(
    identifier="BOOK-BOND",
    start_date=ref,
    maturity_date=ref + "5Y",
    notional=500_000,
    rate=0.045,
    currency=qs.Currency.USD,
    payment_frequency=qs.Frequency.Semiannual,
)
caplet = qs.CapletFloorlet(
    identifier="BOOK-CAPLET",
    fixing_date=ref + "2Y",
    start_accrual_date=ref + "2Y",
    end_accrual_date=ref + "2Y3M",
    payment_date=ref + "2Y3M",
    strike=0.045,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    payoff_type=qs.CapletFloorletType.Caplet,
)
portfolio = [
    qs.Trade(swap, ref, qs.Side.PayShort),
    qs.Trade(bond, ref, pricing_anchor=qs.BondPricingAnchor.z_spread(0.0)),
    qs.Trade(caplet, ref, notional=2_000_000),
]

## Keep identifiers, units, and missing results intact

Request common outputs for all trades and add product-specific requests where
supported. The bond uses its default zero Z-spread here. We save cashflows for
the swap and straight bond; the option example asks for value and sensitivities.
`None` means that a result was not supplied, not that it equals zero.

Risk rows are labelled by quote identifier, so grouping by `pillar` adds
exposure to the *same* input across trades. This is different from summing
absolute risks: offsets between positions should remain visible.

In [ ]:
extra_requests = {
    swap.identifier: [qs.Request.FairRate, qs.Request.Cashflows],
    bond.identifier: [
        qs.Request.YieldToMaturity,
        qs.Request.ModifiedDuration,
        qs.Request.ZSpread,
        qs.Request.Cashflows,
    ],
    caplet.identifier: [],
}
results = []
risk_tables = []
cashflow_tables = []
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    volatility_surfaces=market.volatility_surfaces,
)

with context:
    for trade in portfolio:
        result = context.evaluate(
            trade,
            [
                qs.Request.Value,
                qs.Request.Sensitivities,
                *extra_requests[trade.instrument.identifier],
            ],
        )
        results.append(
            {
                "trade": result.identifier,
                "pv_usd": result.price,
                "fair_rate": result.fair_rate,
                "ytm": result.yield_to_maturity,
                "modified_duration": result.modified_duration,
            }
        )
        risk_tables.append(result.sensitivities.assign(trade=result.identifier))
        if result.cashflows is not None:
            cashflow_tables.append(result.cashflows.assign(trade=result.identifier))

valuation = pd.DataFrame(results).set_index("trade")
trade_risk = pd.concat(
    risk_tables,
    ignore_index=True,
)
portfolio_risk = trade_risk.groupby("pillar")["value"].sum()
portfolio_pv = valuation["pv_usd"].sum()
print(f"Portfolio PV: USD {portfolio_pv:,.2f}")
valuation

## Scale each family of inputs in its own units

The derivative is `dPV/dQuote`, not a pre-scaled delta or vega. A +1 bp rate
move is `1e-4`; a +1 Black-volatility-point move is `0.01`. The independent bond
spread is another rate-like factor. We identify families from the construction
specifications, rather than guessing from a substring such as `SOFR` that is
shared by both rate and volatility labels.

The matrix below preserves each trade's contribution and then adds the signed
total. For a single quote, the P&L approximation is derivative × quote move.

In [ ]:
rate_labels = market.curves[0].to_dict()["quotes"]
vol_labels = market.volatility_surfaces[0].to_dict()["quotes"]
rate_risk = (
    portfolio_risk.reindex(
        rate_labels,
        fill_value=0,
    )
    .rename("dPV_dQuote")
    .to_frame()
)
rate_risk["pnl_plus_1bp_usd"] = rate_risk["dPV_dQuote"] * 1e-4
vol_risk = (
    portfolio_risk.reindex(
        vol_labels,
        fill_value=0,
    )
    .rename("dPV_dQuote")
    .to_frame()
)
vol_risk["pnl_plus_1vol_point_usd"] = vol_risk["dPV_dQuote"] * 0.01
spread_risk = portfolio_risk[portfolio_risk.index.str.endswith(".ZSpread")] * 1e-4

contributions = trade_risk.pivot_table(
    index="pillar",
    columns="trade",
    values="value",
    aggfunc="sum",
).fillna(0)
rate_contributions = (
    contributions.reindex(
        rate_labels,
        fill_value=0,
    )
    * 1e-4
)
rate_contributions["portfolio"] = rate_contributions.sum(axis=1)
rate_contributions


In [ ]:
rate_plot = rate_contributions.drop(columns="portfolio")
rate_plot = rate_plot.loc[rate_plot.abs().max(axis=1) > 1e-8].copy()
rate_plot.index = rate_plot.index.str.replace(
    "FixedRateDeposit_USD_SOFR_",
    "Deposit ",
    regex=False,
).str.replace(
    "OIS_USD_SOFR_",
    "OIS ",
    regex=False,
)
vol_plot = vol_risk.loc[
    vol_risk["pnl_plus_1vol_point_usd"].abs() > 1e-8, "pnl_plus_1vol_point_usd"
]
vol_plot = vol_plot.nlargest(6).copy()
vol_plot.index = (
    vol_plot.index.str.replace(
        "CapletFloorlet_USD_SOFR_3M_",
        "",
        regex=False,
    )
    .str.replace(
        "_Absolute_",
        " @ ",
        regex=False,
    )
    .str.replace(
        "_Straddle_Black",
        "",
        regex=False,
    )
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4),
)
rate_plot.plot.barh(
    stacked=True,
    ax=axes[0],
)
axes[0].set(
    title="Rate-quote risk by position",
    xlabel="USD P&L for +1 bp",
    ylabel="Pillar",
)
axes[0].legend(
    loc="lower left",
    fontsize=8,
)
vol_plot.plot.barh(ax=axes[1])
axes[1].set(
    title="Nonzero volatility-quote risk",
    xlabel="USD P&L for +1 vol point",
    ylabel="Expiry @ strike (3M caplet)",
)
plt.tight_layout()
plt.show()


## Check AAD with a bump and rebuild

Automatic differentiation (AAD) propagates a local PV derivative back through
market construction. It answers a small-move question. Rebuilding under a
scenario answers the actual finite-move question and also captures nonlinearity.

We move all deposit/OIS quotes by the same amount, keeping volatility quotes
and the bond's zero spread unchanged. This is a parallel shift of **input rate
quotes**, not a parallel zero-rate shift. A central difference from +1 bp and
−1 bp should agree closely with the sum of the rate-quote derivatives. The
+100 bp stress can differ noticeably from its linear estimate.

In [ ]:
def portfolio_under_rate_shift(bump):
    scenarios = [
        qs.Scenario(
            label,
            bump,
            qs.ScenarioType.Absolute,
        )
        for label in rate_labels
    ]
    context = qs.PricingContext(
        quotes=market.quotes,
        curves=market.curves,
        csa_policy=market.csa_policy,
        volatility_surfaces=market.volatility_surfaces,
        scenarios=scenarios,
    )

    with context:
        return sum(
            context.evaluate(
                trade,
                [qs.Request.Value],
            ).price
            for trade in portfolio
        )


plus = portfolio_under_rate_shift(1e-4)
minus = portfolio_under_rate_shift(-1e-4)
central_derivative = (plus - minus) / (2e-4)
aad_derivative = rate_risk["dPV_dQuote"].sum()
print("AAD derivative:", aad_derivative)
print("Central-difference derivative:", central_derivative)

stress_pv = portfolio_under_rate_shift(0.01)
pd.DataFrame(
    [
        {
            "shock": "+1 bp",
            "full_pnl_usd": plus - portfolio_pv,
            "linear_pnl_usd": aad_derivative * 1e-4,
        },
        {
            "shock": "+100 bp",
            "full_pnl_usd": stress_pv - portfolio_pv,
            "linear_pnl_usd": aad_derivative * 0.01,
        },
    ]
).assign(
    nonlinear_residual_usd=lambda frame: frame["full_pnl_usd"] - frame["linear_pnl_usd"]
)


## Payment reports are another aggregation, with a different meaning

Group reported contractual cashflows by date and currency to inspect future
receipts and payments. This report includes the swap and bond; it is not an
option-exercise simulation or a complete exposure profile. Discounted PV,
undiscounted payments, duration, and par rates should never be added together.
The latter two are trade-level analytics and are deliberately left as columns
in the valuation table above.

In [ ]:
cashflows = pd.concat(
    cashflow_tables,
    ignore_index=True,
)
cashflows.groupby(["payment_date", "currency"])["amount"].sum().to_frame().head(12)


For reporting, `valuation.to_csv(...)`, `trade_risk.to_csv(...)`, and
`rate_risk.to_csv(...)` retain explicit labels and units. Today's linear
aggregation does not define legal netting: [xva.ipynb](../cva/xva.ipynb) introduces
netting sets, client terms, and future exposure calculations.